In [ ]:

%pip install langchain 
%pip install langchain-community 
%pip install langchain-huggingface
%pip install langchain-chroma 
%pip install chromadb 
%pip install pypdf 
%pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# LangChain

In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "../cleaned_data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={
        "encoding":"utf-8"
    }
)

documents = loader.load()

print("Documents loaded:", len(documents))

Documents loaded: 6


## Text Splitter for Chunking


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ".",
        " ",
        ""
    ]
)

print("Text splitter created successfully!")

Text splitter created successfully!


In [13]:
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 5096


 ## check the chunks

In [14]:
print(chunks[0].page_content)

print("\nMetadata: ")
print(chunks[0].metadata)



UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
x
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
o

Metadata: 
{'source': '..\\cleaned_data\\annual+report_2024.txt'}


## Embedding model 

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":False}
)

print("Embeddings loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14715.35it/s]


Embeddings loaded!


In [17]:
from langchain_chroma import Chroma 

vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../chunks"
)

print("Vector store created successfully!")




Vector store created successfully!


In [18]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_lwargs={"k":3}
)

print("Retriever created !")

Retriever created !


## GEMINI INTEGRATION

In [93]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=api_key)

In [94]:
query = "What did Tesla say about energy storage"

results = retriever.invoke(query)

print("Number of chunks retrieved:", len(results))

Number of chunks retrieved: 4


In [95]:
docs=retriever.invoke(query)

context="\n\n".join(
    doc.page_content
    for doc in docs
)
print(context)

AI is a major pillar of growth for Tesla and the broader economy and key to our pursuit of 
sustainable abundance. Furthermore, AI infrastructure is driving rapid load growth, which, 
along with traditional utility customer applications, is creating an outsized opportunity for our 
Energy storage products to stabilize the grid, shift energy when it is needed most and provide 
additional power capacity. While the current tariff landscape will have a relatively larger impact

developed software to remotely control and dispatch our energy storage systems. Further leveraging our capabilities in AI, every Tesla energy storage
product is capable of being enhanced through firmware updates and optimized by our software platforms, particularly Powerhub (for distributed energy

developed software to remotely control and dispatch our energy storage systems. Further leveraging our capabilities in AI, every Tesla energy storage
product is capable of being enhanced through firmware updates and optim

In [96]:
prompt=f"""
You are a business assistant for Tesla 
Use the context below to answer the user's question.

Context:
{context}

Question:
{query}
Answer in a concise and professional manner.query"""

In [97]:
response=model.generate_content(prompt)
print(response.text)

Unauthenticated: 401 Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project. [reason: "ACCESS_TOKEN_TYPE_UNSUPPORTED"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
metadata {
  key: "method"
  value: "google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent"
}
]